In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import glob
import warnings
warnings.filterwarnings('ignore')

In [2]:
class RegressionDNN(nn.Module):
    """Deep Neural Network for regression with sigmoid output"""
    def __init__(self, input_dim, hidden_dims=[512, 256, 128, 64]):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        layers.append(nn.Sigmoid())  # Sigmoid to ensure output is in [0, 1]
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x).squeeze(-1)

def train_dnn(X_train, y_train, X_test, y_test, epochs=100, lr=0.001, patience=15, verbose=False):
    """Train DNN and return test MAE"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    if verbose:
        print(f"    Training DNN on {device}")
    
    # Normalize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Convert to tensors
    train_dataset = TensorDataset(
        torch.FloatTensor(X_train_scaled),
        torch.FloatTensor(y_train)
    )
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    
    # Initialize model
    model = RegressionDNN(X_train.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.L1Loss()  # MAE loss
    
    # Training loop with early stopping
    model.train()
    best_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        # Early stopping
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter > patience:
                if verbose:
                    print(f"      Early stopping at epoch {epoch}")
                break
    
    # Evaluation
    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
        predictions = model(X_test_tensor).cpu().numpy()
    
    test_mae = mean_absolute_error(y_test, predictions)
    
    if verbose:
        print(f"      Prediction range: [{predictions.min():.3f}, {predictions.max():.3f}]")
        print(f"      All in [0,1]: {np.all((predictions >= 0) & (predictions <= 1))}")
    
    return test_mae, predictions

In [3]:
alphas = np.logspace(-6, 6, 20)

ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ("ridge", RidgeCV(alphas=alphas, cv=5, scoring="neg_mean_absolute_error"))
])

In [4]:
regressions = {
    'xgboost': Pipeline([
        ('scaler', StandardScaler()),
        ('xgb', xgb.XGBRegressor(
            n_estimators=1000, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
        ))
    ]),
    
    'lightGBM': Pipeline([
        ('scaler', StandardScaler()),
        ('lgb', lgb.LGBMRegressor(
            n_estimators=1000, max_depth=8, learning_rate=0.1,
            feature_fraction=0.8, bagging_fraction=0.8, random_state=42, verbosity=-1
        ))
    ]),

    'randomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(
            n_estimators=500, max_depth=15, min_samples_split=5,
            min_samples_leaf=2, random_state=42, n_jobs=-1
        ))
    ]),
    
    'ridge': ridge_pipeline
}


In [5]:
def test_regression(base_folder, regression_name, save=False, output_file=None, visual_only=False):

    files = sorted(glob.glob(base_folder))
    mae_table = {}
    test_maes = []
    dfs = []

    is_dnn = regression_name == 'dnn'
    is_ridge = regression_name == 'ridge'
    if not is_dnn:
        model = regressions[regression_name]

    for fold in range(1, 6):
        fold_folder = files[fold-1]
        print(f'\n Processing Fold {fold}: {fold_folder}')

        if visual_only:
            X_train = pd.read_csv(f"{fold_folder}/X_train_visual.csv").values
            X_test = pd.read_csv(f"{fold_folder}/X_test_visual.csv").values
        else:      
            X_train = pd.read_csv(f"{fold_folder}/X_train_combined.csv").values
            X_test = pd.read_csv(f"{fold_folder}/X_test_combined.csv").values
        y_train = pd.read_csv(f"{fold_folder}/train_predictions_with_metadata.csv")['target'].values
        y_test = pd.read_csv(f"{fold_folder}/test_predictions_with_metadata.csv")['target'].values
    
        test_metadata = pd.read_csv(f"{fold_folder}/test_predictions_with_metadata.csv")[['target', 'CENTROID_ID', 'LATNUM', 'LONGNUM']]
        
        print(f"  Loaded: {X_train.shape[0]} train, {X_test.shape[0]} test samples")

        if is_dnn:
            test_mae, test_predictions = train_dnn(X_train, y_train, X_test, y_test, verbose=True)
        elif is_ridge:
            print("  Performing 5-fold CV for Ridge regression...")
            kf = KFold(n_splits=5, shuffle=True, random_state=42)
            cv_scores = cross_val_score(
                model, X_train, y_train,
                cv=kf, scoring='neg_mean_absolute_error'
            )
            print(f"  CV MAE: {-cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
            
            model.fit(X_train, y_train)
            selected_alpha = model.named_steps['ridge'].alpha_
            print(f"  Selected alpha: {selected_alpha:.6f}")

            test_predictions = model.predict(X_test)
            test_mae = mean_absolute_error(y_test, test_predictions)
        else:
            model.fit(X_train, y_train)
            test_predictions = model.predict(X_test)
            test_mae = mean_absolute_error(y_test, test_predictions)
        
        mae_table[f'fold {fold}'] = test_mae
        test_maes.append(test_mae)
        print(f'{regression_name}: test_mae: {test_mae:.4f}')
    
        fold_df = test_metadata.copy()
        fold_df['prediction'] = test_predictions
        fold_df['error'] = np.abs(y_test - test_predictions)
        fold_df['fold'] = fold
        dfs.append(fold_df)
    
    std_err = np.std(test_maes) / np.sqrt(5)
    print(f'Average: {np.mean(list(mae_table.values()))} ± {std_err}')
        
    
    if save:
        df_combined = pd.concat(dfs, ignore_index=True)
        df_combined.to_csv(f'results/{output_file}', index=False)
        print(f'Data saved to {output_file}')

    return mae_table


## Test Regression Head (sh)

### Combined Features (Geo-encoded + Visual)

In [10]:
base_folder = 'results/split_spatialL_*_sh'
mae_table_sh_xgb = test_regression(base_folder, 'xgboost', save=True, output_file='df_sh_xgb.csv')
mae_table_sh_xgb

['results/split_spatialL_1_sh',
 'results/split_spatialL_2_sh',
 'results/split_spatialL_3_sh',
 'results/split_spatialL_4_sh',
 'results/split_spatialL_5_sh']


 Processing Fold 1: results/split_spatialL_1_loc_locenc
  Loaded: 10602 train, 2713 test samples
xgboost: test_mae: 0.1818

 Processing Fold 2: results/split_spatialL_2_loc_locenc
  Loaded: 10685 train, 2630 test samples
xgboost: test_mae: 0.1772

 Processing Fold 3: results/split_spatialL_3_loc_locenc
  Loaded: 10668 train, 2647 test samples
xgboost: test_mae: 0.1844

 Processing Fold 4: results/split_spatialL_4_loc_locenc
  Loaded: 10645 train, 2670 test samples
xgboost: test_mae: 0.1802

 Processing Fold 5: results/split_spatialL_5_loc_locenc
  Loaded: 10660 train, 2655 test samples
xgboost: test_mae: 0.1850
0.18172151730839073


In [12]:
mae_table_sh_lgb = test_regression(base_folder, 'lightGBM', save=True, output_file='df_sh_lgb.csv')
mae_table_sh_lgb


 Processing Fold 1: results/split_spatialL_1_sh
  Loaded: 10602 train, 2713 test samples
light_gbm: test_mae: 0.1819

 Processing Fold 2: results/split_spatialL_2_sh
  Loaded: 10685 train, 2630 test samples
light_gbm: test_mae: 0.1767

 Processing Fold 3: results/split_spatialL_3_sh
  Loaded: 10668 train, 2647 test samples
light_gbm: test_mae: 0.1820

 Processing Fold 4: results/split_spatialL_4_sh
  Loaded: 10645 train, 2670 test samples
light_gbm: test_mae: 0.1809

 Processing Fold 5: results/split_spatialL_5_sh
  Loaded: 10660 train, 2655 test samples
light_gbm: test_mae: 0.1837
Average: 0.1810708776254164


In [ ]:
mae_table_sh_rf = test_regression(base_folder, 'randomForest', save=True, output_file='df_sh_rf.csv')
mae_table_sh_rf

### Visual-only Features

In [7]:
base_folder = 'results/split_spatialL_*_sh'

mae_table_visual_lgb = test_regression(base_folder, 'lightGBM')
mae_table_visual_lgb


 Processing Fold 1: results/split_spatialL_1_sh
  Loaded: 10602 train, 2713 test samples
lightGBM: test_mae: 0.2134

 Processing Fold 2: results/split_spatialL_2_sh
  Loaded: 10685 train, 2630 test samples
lightGBM: test_mae: 0.2114

 Processing Fold 3: results/split_spatialL_3_sh
  Loaded: 10668 train, 2647 test samples
lightGBM: test_mae: 0.2155

 Processing Fold 4: results/split_spatialL_4_sh
  Loaded: 10645 train, 2670 test samples
lightGBM: test_mae: 0.2116

 Processing Fold 5: results/split_spatialL_5_sh
  Loaded: 10660 train, 2655 test samples
lightGBM: test_mae: 0.2191
Average: 0.21419448467637475 ± 0.001277389471243654


{'fold 1': 0.21340056527483697,
 'fold 2': 0.21142969610238788,
 'fold 3': 0.2155143147630253,
 'fold 4': 0.21155423652626235,
 'fold 5': 0.21907361071536136}

In [8]:
mae_table_visual_ridge = test_regression(base_folder, 'ridge', save=True, output_file='df_visual_ridge.csv')


 Processing Fold 1: results/split_spatialL_1_sh
  Loaded: 10602 train, 2713 test samples
  Performing 5-fold CV for Ridge regression...
  CV MAE: 0.2012 ± 0.0029
  Selected alpha: 162.377674
ridge: test_mae: 0.2146

 Processing Fold 2: results/split_spatialL_2_sh
  Loaded: 10685 train, 2630 test samples
  Performing 5-fold CV for Ridge regression...
  CV MAE: 0.1980 ± 0.0025
  Selected alpha: 695.192796
ridge: test_mae: 0.2129

 Processing Fold 3: results/split_spatialL_3_sh
  Loaded: 10668 train, 2647 test samples
  Performing 5-fold CV for Ridge regression...
  CV MAE: 0.1960 ± 0.0039
  Selected alpha: 695.192796
ridge: test_mae: 0.2145

 Processing Fold 4: results/split_spatialL_4_sh
  Loaded: 10645 train, 2670 test samples
  Performing 5-fold CV for Ridge regression...
  CV MAE: 0.1921 ± 0.0023
  Selected alpha: 695.192796
ridge: test_mae: 0.2149

 Processing Fold 5: results/split_spatialL_5_sh
  Loaded: 10660 train, 2655 test samples
  Performing 5-fold CV for Ridge regression...


 Processing Fold 1: results/split_spatialL_1_loc_locenc
  Loaded: 10602 train, 2713 test samples
light_gbm: test_mae: 0.2134

 Processing Fold 2: results/split_spatialL_2_loc_locenc
  Loaded: 10685 train, 2630 test samples
light_gbm: test_mae: 0.2114

 Processing Fold 3: results/split_spatialL_3_loc_locenc
  Loaded: 10668 train, 2647 test samples
light_gbm: test_mae: 0.2155

 Processing Fold 4: results/split_spatialL_4_loc_locenc
  Loaded: 10645 train, 2670 test samples
light_gbm: test_mae: 0.2116

 Processing Fold 5: results/split_spatialL_5_loc_locenc
  Loaded: 10660 train, 2655 test samples
light_gbm: test_mae: 0.2191
Average: 0.21419448467637475


In [15]:
mae_table_visual_xgb = test_regression(base_folder, 'xgboost')
mae_table_visual_xgb


 Processing Fold 1: results/split_spatialL_1_loc_locenc
  Loaded: 10602 train, 2713 test samples
xgboost: test_mae: 0.2137

 Processing Fold 2: results/split_spatialL_2_loc_locenc
  Loaded: 10685 train, 2630 test samples
xgboost: test_mae: 0.2130

 Processing Fold 3: results/split_spatialL_3_loc_locenc
  Loaded: 10668 train, 2647 test samples
xgboost: test_mae: 0.2175

 Processing Fold 4: results/split_spatialL_4_loc_locenc
  Loaded: 10645 train, 2670 test samples
xgboost: test_mae: 0.2126

 Processing Fold 5: results/split_spatialL_5_loc_locenc
  Loaded: 10660 train, 2655 test samples
xgboost: test_mae: 0.2186
0.21508324470107493


## Test Regression Head (sh+siren)

In [7]:
base_folder = 'results/split_spatialL_*_sh_siren'
files = sorted(glob.glob(base_folder))
files

['results/split_spatialL_1_sh_siren',
 'results/split_spatialL_2_sh_siren',
 'results/split_spatialL_3_sh_siren',
 'results/split_spatialL_4_sh_siren',
 'results/split_spatialL_5_sh_siren']

In [8]:
mae_table_sh_siren_lgb = test_regression(base_folder, 'lightGBM')
mae_table_sh_siren_lgb


 Processing Fold 1: results/split_spatialL_1_sh_siren
  Loaded: 10602 train, 2713 test samples
lightGBM: test_mae: 0.2057

 Processing Fold 2: results/split_spatialL_2_sh_siren
  Loaded: 10685 train, 2630 test samples
lightGBM: test_mae: 0.2078

 Processing Fold 3: results/split_spatialL_3_sh_siren
  Loaded: 10668 train, 2647 test samples
lightGBM: test_mae: 0.2116

 Processing Fold 4: results/split_spatialL_4_sh_siren
  Loaded: 10645 train, 2670 test samples
lightGBM: test_mae: 0.2065

 Processing Fold 5: results/split_spatialL_5_sh_siren
  Loaded: 10660 train, 2655 test samples
lightGBM: test_mae: 0.2109
Average: 0.20849694050105572 ± 0.0010498811882847482


{'fold 1': 0.20570945908077848,
 'fold 2': 0.2077622822980258,
 'fold 3': 0.21161990039624443,
 'fold 4': 0.20652697549607418,
 'fold 5': 0.21086608523415581}

In [9]:
mae_table_sh_siren_xgb = test_regression(base_folder, 'xgboost')
mae_table_sh_siren_xgb


 Processing Fold 1: results/split_spatialL_1_sh_siren
  Loaded: 10602 train, 2713 test samples
xgboost: test_mae: 0.2084

 Processing Fold 2: results/split_spatialL_2_sh_siren
  Loaded: 10685 train, 2630 test samples
xgboost: test_mae: 0.2094

 Processing Fold 3: results/split_spatialL_3_sh_siren
  Loaded: 10668 train, 2647 test samples
xgboost: test_mae: 0.2121

 Processing Fold 4: results/split_spatialL_4_sh_siren
  Loaded: 10645 train, 2670 test samples
xgboost: test_mae: 0.2087

 Processing Fold 5: results/split_spatialL_5_sh_siren
  Loaded: 10660 train, 2655 test samples
xgboost: test_mae: 0.2131
Average: 0.21034912345578222 ± 0.0008467356045926099


{'fold 1': 0.20835388446193842,
 'fold 2': 0.20944830802678446,
 'fold 3': 0.212141805968414,
 'fold 4': 0.20874336217570041,
 'fold 5': 0.21305825664607378}

In [10]:
mae_table_sh_siren_rf = test_regression(base_folder, 'randomForest')
mae_table_sh_siren_rf


 Processing Fold 1: results/split_spatialL_1_sh_siren
  Loaded: 10602 train, 2713 test samples
randomForest: test_mae: 0.2097

 Processing Fold 2: results/split_spatialL_2_sh_siren
  Loaded: 10685 train, 2630 test samples
randomForest: test_mae: 0.2115

 Processing Fold 3: results/split_spatialL_3_sh_siren
  Loaded: 10668 train, 2647 test samples
randomForest: test_mae: 0.2149

 Processing Fold 4: results/split_spatialL_4_sh_siren
  Loaded: 10645 train, 2670 test samples
randomForest: test_mae: 0.2125

 Processing Fold 5: results/split_spatialL_5_sh_siren
  Loaded: 10660 train, 2655 test samples
randomForest: test_mae: 0.2152
Average: 0.21276898075774858 ± 0.0009294913677765988


{'fold 1': 0.20971525031282767,
 'fold 2': 0.21147983148967578,
 'fold 3': 0.21487770266556755,
 'fold 4': 0.21253157803071845,
 'fold 5': 0.21524054128995365}

In [11]:
mae_table_sh_siren_dnn = test_regression(base_folder, 'dnn')
mae_table_sh_siren_dnn


 Processing Fold 1: results/split_spatialL_1_sh_siren
  Loaded: 10602 train, 2713 test samples
    Training DNN on cpu
      Prediction range: [0.036, 1.000]
      All in [0,1]: True
dnn: test_mae: 0.2112

 Processing Fold 2: results/split_spatialL_2_sh_siren
  Loaded: 10685 train, 2630 test samples
    Training DNN on cpu
      Prediction range: [0.048, 1.000]
      All in [0,1]: True
dnn: test_mae: 0.2095

 Processing Fold 3: results/split_spatialL_3_sh_siren
  Loaded: 10668 train, 2647 test samples
    Training DNN on cpu
      Prediction range: [0.041, 1.000]
      All in [0,1]: True
dnn: test_mae: 0.2155

 Processing Fold 4: results/split_spatialL_4_sh_siren
  Loaded: 10645 train, 2670 test samples
    Training DNN on cpu
      Prediction range: [0.033, 1.000]
      All in [0,1]: True
dnn: test_mae: 0.2079

 Processing Fold 5: results/split_spatialL_5_sh_siren
  Loaded: 10660 train, 2655 test samples
    Training DNN on cpu
      Prediction range: [0.031, 1.000]
      All in [0,1

{'fold 1': 0.21120561628367518,
 'fold 2': 0.20948076104907926,
 'fold 3': 0.21551491858506816,
 'fold 4': 0.20786739877775143,
 'fold 5': 0.22197566873635574}

## Cleaned Data (sh)

In [6]:
base_folder = 'results/split_spatialL_*_sh_cleaned'

In [7]:
mae_table_sh_cleaned_lgb = test_regression(base_folder, 'lightGBM', True, 'df_sh_cleaned_lgb.csv')
mae_table_sh_cleaned_lgb


 Processing Fold 1: results/split_spatialL_1_sh_cleaned
  Loaded: 18178 train, 4531 test samples
lightGBM: test_mae: 0.1876

 Processing Fold 2: results/split_spatialL_2_sh_cleaned
  Loaded: 18200 train, 4509 test samples
lightGBM: test_mae: 0.1919

 Processing Fold 3: results/split_spatialL_3_sh_cleaned
  Loaded: 18201 train, 4508 test samples
lightGBM: test_mae: 0.1919

 Processing Fold 4: results/split_spatialL_4_sh_cleaned
  Loaded: 18202 train, 4507 test samples
lightGBM: test_mae: 0.1921

 Processing Fold 5: results/split_spatialL_5_sh_cleaned
  Loaded: 18055 train, 4654 test samples
lightGBM: test_mae: 0.1883
Average: 0.19035354418178194 ± 0.0008878109585756896
Data saved to df_sh_cleaned_lgb.csv


{'fold 1': 0.18759663582203862,
 'fold 2': 0.19185317750989184,
 'fold 3': 0.19190198490481128,
 'fold 4': 0.1921335399801177,
 'fold 5': 0.18828238269205017}

In [8]:
mae_table_sh_cleaned_xgb = test_regression(base_folder, 'xgboost', True, 'df_sh_cleaned_xgb.csv')
mae_table_sh_cleaned_xgb


 Processing Fold 1: results/split_spatialL_1_sh_cleaned
  Loaded: 18178 train, 4531 test samples
xgboost: test_mae: 0.1877

 Processing Fold 2: results/split_spatialL_2_sh_cleaned
  Loaded: 18200 train, 4509 test samples
xgboost: test_mae: 0.1929

 Processing Fold 3: results/split_spatialL_3_sh_cleaned
  Loaded: 18201 train, 4508 test samples
xgboost: test_mae: 0.1925

 Processing Fold 4: results/split_spatialL_4_sh_cleaned
  Loaded: 18202 train, 4507 test samples
xgboost: test_mae: 0.1912

 Processing Fold 5: results/split_spatialL_5_sh_cleaned
  Loaded: 18055 train, 4654 test samples
xgboost: test_mae: 0.1882
Average: 0.19050523836196748 ± 0.0009671387615676981
Data saved to df_sh_cleaned_xgb.csv


{'fold 1': 0.18768154414805865,
 'fold 2': 0.19293981511291242,
 'fold 3': 0.19246052116149673,
 'fold 4': 0.19121110731754623,
 'fold 5': 0.18823320406982333}

In [9]:
mae_table_sh_cleaned_rf = test_regression(base_folder, 'randomForest', True, 'df_sh_cleaned_rf.csv')
mae_table_sh_cleaned_rf


 Processing Fold 1: results/split_spatialL_1_sh_cleaned
  Loaded: 18178 train, 4531 test samples
randomForest: test_mae: 0.1917

 Processing Fold 2: results/split_spatialL_2_sh_cleaned
  Loaded: 18200 train, 4509 test samples
randomForest: test_mae: 0.1965

 Processing Fold 3: results/split_spatialL_3_sh_cleaned
  Loaded: 18201 train, 4508 test samples
randomForest: test_mae: 0.1989

 Processing Fold 4: results/split_spatialL_4_sh_cleaned
  Loaded: 18202 train, 4507 test samples
randomForest: test_mae: 0.1974

 Processing Fold 5: results/split_spatialL_5_sh_cleaned
  Loaded: 18055 train, 4654 test samples
randomForest: test_mae: 0.1947
Average: 0.1958423425633105 ± 0.0011101427245960133
Data saved to df_sh_cleaned_rf.csv


{'fold 1': 0.19170739390756736,
 'fold 2': 0.19654638961510643,
 'fold 3': 0.19890433645518307,
 'fold 4': 0.19740031126158342,
 'fold 5': 0.19465328157711223}

In [10]:
mae_table_sh_cleaned_dnn = test_regression(base_folder, 'dnn', True, 'df_sh_cleaned_dnn.csv')


 Processing Fold 1: results/split_spatialL_1_sh_cleaned
  Loaded: 18178 train, 4531 test samples
    Training DNN on cpu
      Prediction range: [0.001, 1.000]
      All in [0,1]: True
dnn: test_mae: 0.1957

 Processing Fold 2: results/split_spatialL_2_sh_cleaned
  Loaded: 18200 train, 4509 test samples
    Training DNN on cpu
      Prediction range: [0.015, 1.000]
      All in [0,1]: True
dnn: test_mae: 0.2022

 Processing Fold 3: results/split_spatialL_3_sh_cleaned
  Loaded: 18201 train, 4508 test samples
    Training DNN on cpu
      Prediction range: [0.005, 1.000]
      All in [0,1]: True
dnn: test_mae: 0.2018

 Processing Fold 4: results/split_spatialL_4_sh_cleaned
  Loaded: 18202 train, 4507 test samples
    Training DNN on cpu
      Prediction range: [0.004, 1.000]
      All in [0,1]: True
dnn: test_mae: 0.2002

 Processing Fold 5: results/split_spatialL_5_sh_cleaned
  Loaded: 18055 train, 4654 test samples
    Training DNN on cpu
      Prediction range: [0.005, 1.000]
      A

### Visual-only

In [ ]:
mae_table_sh_visual_cleaned_lgb = test_regression(base_folder, 'randomForest', visual_only=True)
mae_table_sh_visual_cleaned_lgb

## Cleaned Data (sh+siren)

In [6]:
base_folder = 'results/split_spatialL_*_sh_siren_ccleaned'
glob.glob(base_folder)

['results/split_spatialL_2_sh_siren_cleaned',
 'results/split_spatialL_5_sh_siren_cleaned',
 'results/split_spatialL_3_sh_siren_cleaned',
 'results/split_spatialL_4_sh_siren_cleaned',
 'results/split_spatialL_1_sh_siren_cleaned']

In [7]:
mae_table_sh_siren_cleaned_lgb = test_regression(base_folder, 'lightGBM')
mae_table_sh_siren_cleaned_lgb


 Processing Fold 1: results/split_spatialL_1_sh_siren_cleaned
  Loaded: 11576 train, 2968 test samples
lightGBM: test_mae: 0.2078

 Processing Fold 2: results/split_spatialL_2_sh_siren_cleaned
  Loaded: 11623 train, 2921 test samples
lightGBM: test_mae: 0.2124

 Processing Fold 3: results/split_spatialL_3_sh_siren_cleaned
  Loaded: 11647 train, 2897 test samples
lightGBM: test_mae: 0.2146

 Processing Fold 4: results/split_spatialL_4_sh_siren_cleaned
  Loaded: 11627 train, 2917 test samples
lightGBM: test_mae: 0.2108

 Processing Fold 5: results/split_spatialL_5_sh_siren_cleaned
  Loaded: 11703 train, 2841 test samples
lightGBM: test_mae: 0.2090
Average: 0.2109094234749665 ± 0.0010789788491384173


{'fold 1': 0.20777541863289745,
 'fold 2': 0.21243706050825603,
 'fold 3': 0.2145637670185548,
 'fold 4': 0.21075085274018165,
 'fold 5': 0.20902001847494245}

In [8]:
mae_table_sh_siren_cleaned_xgb = test_regression(base_folder, 'xgboost')
mae_table_sh_siren_cleaned_xgb


 Processing Fold 1: results/split_spatialL_1_sh_siren_cleaned
  Loaded: 11576 train, 2968 test samples
xgboost: test_mae: 0.2078

 Processing Fold 2: results/split_spatialL_2_sh_siren_cleaned
  Loaded: 11623 train, 2921 test samples
xgboost: test_mae: 0.2135

 Processing Fold 3: results/split_spatialL_3_sh_siren_cleaned
  Loaded: 11647 train, 2897 test samples
xgboost: test_mae: 0.2157

 Processing Fold 4: results/split_spatialL_4_sh_siren_cleaned
  Loaded: 11627 train, 2917 test samples
xgboost: test_mae: 0.2103

 Processing Fold 5: results/split_spatialL_5_sh_siren_cleaned
  Loaded: 11703 train, 2841 test samples
xgboost: test_mae: 0.2071
Average: 0.21088863167958466 ± 0.00146504313528941


{'fold 1': 0.20783769066233743,
 'fold 2': 0.21345336307315568,
 'fold 3': 0.21571299978846328,
 'fold 4': 0.2103146009195135,
 'fold 5': 0.20712450395445334}

In [9]:
mae_table_sh_siren_cleaned_rf = test_regression(base_folder, 'randomForest')
mae_table_sh_siren_cleaned_rf


 Processing Fold 1: results/split_spatialL_1_sh_siren_cleaned
  Loaded: 11576 train, 2968 test samples
randomForest: test_mae: 0.2118

 Processing Fold 2: results/split_spatialL_2_sh_siren_cleaned
  Loaded: 11623 train, 2921 test samples
randomForest: test_mae: 0.2133

 Processing Fold 3: results/split_spatialL_3_sh_siren_cleaned
  Loaded: 11647 train, 2897 test samples
randomForest: test_mae: 0.2196

 Processing Fold 4: results/split_spatialL_4_sh_siren_cleaned
  Loaded: 11627 train, 2917 test samples
randomForest: test_mae: 0.2135

 Processing Fold 5: results/split_spatialL_5_sh_siren_cleaned
  Loaded: 11703 train, 2841 test samples
randomForest: test_mae: 0.2126
Average: 0.2141726446548058 ± 0.0012391486718889794


{'fold 1': 0.21181883254146266,
 'fold 2': 0.21332880000165405,
 'fold 3': 0.21958746331609094,
 'fold 4': 0.21348629939045677,
 'fold 5': 0.2126418280243645}

In [10]:
mae_table_sh_siren_cleaned_dnn = test_regression(base_folder, 'dnn')
mae_table_sh_siren_cleaned_dnn


 Processing Fold 1: results/split_spatialL_1_sh_siren_cleaned
  Loaded: 11576 train, 2968 test samples
    Training DNN on cpu
      Prediction range: [0.003, 1.000]
      All in [0,1]: True
dnn: test_mae: 0.2075

 Processing Fold 2: results/split_spatialL_2_sh_siren_cleaned
  Loaded: 11623 train, 2921 test samples
    Training DNN on cpu
      Prediction range: [0.005, 1.000]
      All in [0,1]: True
dnn: test_mae: 0.2189

 Processing Fold 3: results/split_spatialL_3_sh_siren_cleaned
  Loaded: 11647 train, 2897 test samples
    Training DNN on cpu
      Prediction range: [0.014, 1.000]
      All in [0,1]: True
dnn: test_mae: 0.2179

 Processing Fold 4: results/split_spatialL_4_sh_siren_cleaned
  Loaded: 11627 train, 2917 test samples
    Training DNN on cpu
      Prediction range: [0.004, 1.000]
      All in [0,1]: True
dnn: test_mae: 0.2137

 Processing Fold 5: results/split_spatialL_5_sh_siren_cleaned
  Loaded: 11703 train, 2841 test samples
    Training DNN on cpu
      Prediction

{'fold 1': 0.20748336979840698,
 'fold 2': 0.2189492547168107,
 'fold 3': 0.21794781374181735,
 'fold 4': 0.213725118996232,
 'fold 5': 0.20822317292164189}